In [2]:
# A. Membaca dan Eksplorasi Awal
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Tugas4") \
    .master("local[*]") \
    .getOrCreate()

df = spark.read.csv(
    "hdfs://localhost:9000/user/mahasiswa/tugas4/transaksi_september_2026.csv",
    header=True,
    inferSchema=True
)

df.printSchema()

print("Jumlah baris:", df.count())

df.show(10)

root
 |-- order_id: string (nullable = true)
 |-- tanggal: timestamp (nullable = true)
 |-- kategori: string (nullable = true)
 |-- kota: string (nullable = true)
 |-- unit_terjual: integer (nullable = true)
 |-- harga_satuan: integer (nullable = true)
 |-- metode_pembayaran: string (nullable = true)
 |-- rating: double (nullable = true)

Jumlah baris: 1000
+--------+-------------------+--------------------+----------+------------+------------+-----------------+------+
|order_id|            tanggal|            kategori|      kota|unit_terjual|harga_satuan|metode_pembayaran|rating|
+--------+-------------------+--------------------+----------+------------+------------+-----------------+------+
|ORD-3000|2026-09-02 00:00:00|        Rumah Tangga|Yogyakarta|           3|       90000|              COD|   4.0|
|ORD-3001|2026-09-04 00:00:00|   Makanan & Minuman|      Solo|           3|      200000|         E-Wallet|   5.0|
|ORD-3002|2026-09-26 00:00:00|Kesehatan & Kecan...|  Semarang|        

In [28]:
#B. Mengenai Data Kosong
from pyspark.sql.functions import col

# Menghitung jumlah rating yang kosong
jumlah_kosong = df.filter(col("rating").isNull()).count()
print("Jumlah rating kosong:", jumlah_kosong)

# Menangani missing value dengan df.na.fill()
df = df.na.fill({"rating": 0})

# Cek kembali setelah diisi
print("Jumlah rating kosong setelah fill:", df.filter(col("rating").isNull()).count())

Jumlah rating kosong: 0
Jumlah rating kosong setelah fill: 0


In [29]:
#C. Transformasi Data
from pyspark.sql.functions import col,when
#Menambah kolom total_pendapatan
df = df.withColumn(
    "total_pendapatan",
    col("unit_terjual") * col("harga_satuan")
)

#Menambah kolom tier_transaksi
df = df.withColumn(
    "tier_transaksi",
    when(col("total_pendapatan") > 500000, "Besar")
    .otherwise("Kecil")
)

df.select("order_id", "total_pendapatan","tier_transaksi").show(10)

+--------+----------------+--------------+
|order_id|total_pendapatan|tier_transaksi|
+--------+----------------+--------------+
|ORD-3000|          270000|         Kecil|
|ORD-3001|          600000|         Besar|
|ORD-3002|          480000|         Kecil|
|ORD-3003|         2100000|         Besar|
|ORD-3004|          600000|         Besar|
|ORD-3005|          100000|         Kecil|
|ORD-3006|           40000|         Kecil|
|ORD-3007|          720000|         Besar|
|ORD-3008|          140000|         Kecil|
|ORD-3009|          900000|         Besar|
+--------+----------------+--------------+
only showing top 10 rows



In [17]:
#D. Analisis dengan GroupBy
from pyspark.sql.functions import col, sum, avg, desc
#1. Kategori dengan total pendapatan tertinggi
kategori_tertinggi = df.groupBy("kategori") \
    .agg(sum("total_pendapatan").alias("total_pendapatan")) \
    .orderBy(desc("total_pendapatan"))
kategori_tertinggi.show(1)

#2. Kota dengan jumlah transaksi tier "Besar" terbanyak
kota_besar = df.filter(col("tier_transaksi") == "Besar") \
    .groupBy("kota") \
    .count() \
    .orderBy(desc("count"))
kota_besar.show(1)

#3. Rata-rata rating tiap metode pembayaran
rating_metode = df.groupBy("metode_pembayaran") \
    .agg(avg("rating").alias("rata_rating")) \
    .orderBy(desc("rata_rating"))

rating_metode.show(1)

+------------+----------------+
|    kategori|total_pendapatan|
+------------+----------------+
|Rumah Tangga|       138665000|
+------------+----------------+
only showing top 1 row

+----+-----+
|kota|count|
+----+-----+
|Solo|   92|
+----+-----+
only showing top 1 row

+-----------------+------------------+
|metode_pembayaran|       rata_rating|
+-----------------+------------------+
|              COD|3.3745019920318726|
+-----------------+------------------+
only showing top 1 row



In [30]:
#E.Menyimpan DataFrame hasil bagian C ke HDFS DALAM FORMAT CSV baru
output_path = "hdfs://localhost:9000/user/mahasiswa/tugas4/hasil_transaksi"

df.write \
    .mode("overwrite") \
    .option("header", True) \
    .csv(output_path)

print("Data berhasil disimpan ke HDFS")

Data berhasil disimpan ke HDFS


In [31]:
#verifikasi hasil penyimpanan
!hdfs dfs -ls /user/mahasiswa/tugas4/hasil_transaksi

Found 2 items
-rw-r--r--   3 Ziffy2 supergroup          0 2026-09-16 07:57 /user/mahasiswa/tugas4/hasil_transaksi/_SUCCESS
-rw-r--r--   3 Ziffy2 supergroup      92296 2026-09-16 07:57 /user/mahasiswa/tugas4/hasil_transaksi/part-00000-1dc2ab4c-d502-4f34-81fc-73e5c8aa1be4-c000.csv


In [45]:
# Membuat atau membuka kembali SparkSession
spark = SparkSession.builder \
    .appName("NamaAplikasiAnda") \
    .getOrCreate()

print("SparkSession dibuka.")

SparkSession dibuka.


In [ ]:
df = spark.read.csv(
     "hdfs://localhost:9000/user/mahasiswa/tugas4/transaksi_september_2026.csv",
    header=True,
    inferSchema=True
)

In [50]:
df.describe().show()


+-------+--------+------------+----------+----------------+------------------+-----------------+------------------+
|summary|order_id|    kategori|      kota|    unit_terjual|      harga_satuan|metode_pembayaran|            rating|
+-------+--------+------------+----------+----------------+------------------+-----------------+------------------+
|  count|    1000|        1000|      1000|            1000|              1000|             1000|               796|
|   mean|    NULL|        NULL|      NULL|           5.929|          127550.0|             NULL|4.1457286432160805|
| stddev|    NULL|        NULL|      NULL|3.16543503787062|106042.40893384359|             NULL|0.9668046815725551|
|    min|ORD-3000|  Elektronik|   Kebumen|               1|             20000|              COD|               1.0|
|    max|ORD-3999|Rumah Tangga|Yogyakarta|              11|            350000|    Transfer Bank|               5.0|
+-------+--------+------------+----------+----------------+-------------

In [55]:
df.show(10, truncate=False)

+--------+-------------------+----------------------+----------+------------+------------+-----------------+------+
|order_id|tanggal            |kategori              |kota      |unit_terjual|harga_satuan|metode_pembayaran|rating|
+--------+-------------------+----------------------+----------+------------+------------+-----------------+------+
|ORD-3000|2026-09-02 00:00:00|Rumah Tangga          |Yogyakarta|3           |90000       |COD              |4.0   |
|ORD-3001|2026-09-04 00:00:00|Makanan & Minuman     |Solo      |3           |200000      |E-Wallet         |5.0   |
|ORD-3002|2026-09-26 00:00:00|Kesehatan & Kecantikan|Semarang  |8           |60000       |E-Wallet         |3.0   |
|ORD-3003|2026-09-09 00:00:00|Makanan & Minuman     |Semarang  |6           |350000      |Transfer Bank    |4.0   |
|ORD-3004|2026-09-10 00:00:00|Rumah Tangga          |Yogyakarta|10          |60000       |E-Wallet         |4.0   |
|ORD-3005|2026-09-09 00:00:00|Fashion               |Purworejo |5       

In [56]:
df.groupBy("kota") \
  .count() \
  .orderBy(desc("count")) \
  .show()

+----------+-----+
|      kota|count|
+----------+-----+
|      Solo|  189|
|Yogyakarta|  175|
|   Kebumen|  164|
|  Magelang|  162|
|  Semarang|  158|
| Purworejo|  152|
+----------+-----+



In [57]:
df.groupBy("metode_pembayaran") \
  .count() \
  .orderBy(desc("count")) \
  .show()

+-----------------+-----+
|metode_pembayaran|count|
+-----------------+-----+
|    Transfer Bank|  253|
|              COD|  251|
|         E-Wallet|  250|
|     Kartu Kredit|  246|
+-----------------+-----+

